# Pertemuan 13 — Deep Learning & NLP Dasar

**Nama:** Amin Siddik Rangkuti  
**NIM:** 220401010124  
**Program Studi:** Informatika  

Notebook ini berisi aktivitas hands-on Pertemuan 13:
1. Klasifikasi non-linear menggunakan Neural Network.
2. Analisis sentimen sederhana menggunakan TF-IDF dan Logistic Regression.


## Langkah 1 — Generate & Eksplorasi Dataset Non-Linear

Dataset `make_moons` digunakan karena memiliki dua kelas berbentuk bulan sabit yang saling melengkung, sehingga tidak mudah dipisahkan hanya dengan satu garis lurus.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=300, noise=0.2, random_state=42)

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', alpha=0.7)
plt.title('Dataset Non-Linear (2 Kelas Bulan Sabit)')
plt.xlabel('Fitur 1')
plt.ylabel('Fitur 2')
plt.show()


**Interpretasi:**  
Dua kelas pada dataset terlihat saling melengkung dan bertautan. Karena pola pemisahnya tidak linear, neural network dengan activation function non-linear lebih sesuai digunakan dibandingkan model yang hanya menghasilkan batas keputusan berupa garis lurus.


## Langkah 2 — Bangun & Latih Neural Network Sederhana

Arsitektur yang digunakan:
- Hidden layer 1: 16 neuron, ReLU
- Hidden layer 2: 8 neuron, ReLU
- Output layer: 1 neuron, Sigmoid


In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = Sequential([
    Dense(16, activation='relu', input_shape=(2,)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    X_tr, y_tr,
    epochs=30,
    validation_split=0.2,
    verbose=0
)

print("Training selesai.")


## Langkah 3 — Evaluasi & Visualisasi Kurva Belajar


In [ ]:
loss, acc = model.evaluate(X_te, y_te, verbose=0)
print(f'Akurasi pada data uji: {acc:.3f}')

plt.figure(figsize=(7, 5))
plt.plot(history.history['accuracy'], label='Training')
plt.plot(history.history['val_accuracy'], label='Validasi')
plt.xlabel('Epoch')
plt.ylabel('Akurasi')
plt.title('Kurva Pembelajaran Model')
plt.legend()
plt.show()


**Interpretasi:**  
Perhatikan kurva training dan validasi setelah notebook dijalankan. Jika keduanya meningkat dan nilainya relatif berdekatan, model belajar dengan baik. Jika akurasi training terus naik tetapi akurasi validasi tertinggal atau menurun jauh, hal tersebut dapat menjadi indikasi overfitting.


## Langkah 4 — Siapkan Dataset Ulasan Produk

Dataset berikut berisi 40 ulasan sintetis berbahasa Indonesia dengan label:
- `1` = positif
- `0` = negatif


In [ ]:
ulasan = [
    'Barangnya bagus banget, pengiriman cepat',
    'Kualitas jelek, tidak sesuai deskripsi',
    'Sangat puas, akan beli lagi',
    'Kecewa, barang rusak saat sampai',
    'Recommended, harga sesuai kualitas',
    'Buruk sekali, tidak sesuai ekspektasi',
    'Produknya sangat bagus dan bermanfaat',
    'Pengiriman sangat lama dan mengecewakan',
    'Kualitas produk memuaskan',
    'Barang cacat dan tidak bisa digunakan',
    'Pelayanan ramah dan cepat',
    'Produk tidak sesuai dengan foto',
    'Harga murah tetapi kualitas bagus',
    'Saya menyesal membeli produk ini',
    'Barang sampai dengan aman dan rapi',
    'Kualitas sangat buruk',
    'Produk bekerja dengan baik',
    'Paket datang terlambat',
    'Sangat suka dengan produknya',
    'Barang mudah rusak',
    'Penjual responsif dan ramah',
    'Produk mengecewakan',
    'Hasilnya sesuai harapan',
    'Tidak direkomendasikan',
    'Barang original dan berkualitas',
    'Pelayanan sangat buruk',
    'Pengemasan sangat rapi',
    'Produk tidak berfungsi',
    'Harga sesuai dengan kualitas',
    'Saya tidak puas dengan barangnya',
    'Barang bagus dan cepat sampai',
    'Deskripsi produk tidak sesuai',
    'Kualitasnya sangat memuaskan',
    'Barang rusak setelah digunakan',
    'Sangat recommended untuk dibeli',
    'Pengiriman mengecewakan dan lambat',
    'Produk sesuai pesanan',
    'Kualitas jauh dari harapan',
    'Saya puas dengan pelayanan penjual',
    'Barang tidak sesuai ekspektasi'
]

label = [
    1,0,1,0,1,0,1,0,1,0,
    1,0,1,0,1,0,1,0,1,0,
    1,0,1,0,1,0,1,0,1,0,
    1,0,1,0,1,0,1,0,1,0
]

print('Jumlah ulasan:', len(ulasan))
print('Jumlah label:', len(label))


## Langkah 5 — Ubah Teks Menjadi TF-IDF

TF-IDF memberi bobot lebih tinggi pada kata yang lebih khas/bermakna dalam dokumen dan bobot lebih rendah pada kata yang terlalu umum.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_text = tfidf.fit_transform(ulasan)

print('Jumlah kata unik:', len(tfidf.get_feature_names_out()))
print('10 kata pertama:')
print(tfidf.get_feature_names_out()[:10])


## Langkah 6 — Latih Model Klasifikasi Sentimen & Evaluasi


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

Xt_tr, Xt_te, yt_tr, yt_te = train_test_split(
    X_text, label, test_size=0.2, random_state=42
)

model_sentimen = LogisticRegression()
model_sentimen.fit(Xt_tr, yt_tr)

akurasi = model_sentimen.score(Xt_te, yt_te)
print(f'Akurasi model sentimen: {akurasi:.3f}')


In [ ]:
kalimat_baru = ['Pelayanan sangat memuaskan dan ramah']
pred = model_sentimen.predict(tfidf.transform(kalimat_baru))

print('Kalimat:', kalimat_baru[0])
print('Prediksi:', 'Positif' if pred[0] == 1 else 'Negatif')


## Kesimpulan

Pada praktikum ini, neural network digunakan untuk menyelesaikan klasifikasi data non-linear dengan memanfaatkan hidden layer dan activation function ReLU serta Sigmoid. Model dilatih menggunakan proses iteratif selama beberapa epoch dan dievaluasi menggunakan data uji.

Pada bagian NLP, teks ulasan diubah menjadi representasi numerik menggunakan TF-IDF. Representasi tersebut kemudian digunakan oleh Logistic Regression untuk melakukan klasifikasi sentimen positif dan negatif. TF-IDF cocok digunakan sebagai pendekatan awal karena sederhana, cepat, dan mudah dipahami.


## Link GitHub

Setelah notebook ini selesai dijalankan di Google Colab, upload file `.ipynb` ke repository GitHub Anda. Kemudian salin URL file notebook tersebut dan kirimkan pada Forum Diskusi 13 di LMS.
